# 00 — Environment and Preflight

**Purpose:** establish the persistent project path, extract the ZIP once, check out the exact upstream NCM code revision, install the project's declared dependencies, and verify the Colab GPU. This notebook does **not** download the NCM weights or run inference.

**Before starting:** upload `NCM_Research_Colab_Project.zip` to the root of Google Drive (`MyDrive`). From the ZIP on your computer, also open/upload this notebook in Colab (or open the extracted copy from Drive). The ZIP itself is needed by the first bootstrap cell. Select a Colab GPU runtime before continuing.

The official NCM code is cloned into the ephemeral `/content/ncm-upstream` directory at a pinned commit. Checkpoints, source records, SQLite state, and export artifacts live under the persistent `NCM Research/local_outputs/` folder on Drive.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, zipfile
from google.colab import drive

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_PARENT = DRIVE_ROOT / 'NCM Research'
ZIP_PATH = DRIVE_ROOT / 'NCM_Research_Colab_Project.zip'
drive.mount('/content/drive')

if not (PROJECT_PARENT / 'github' / 'pyproject.toml').exists():
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(
            f"Upload the downloaded ZIP to MyDrive as {ZIP_PATH.name}; it was not found at {ZIP_PATH}. "
            "Do not unzip into an unrelated folder."
        )
    # Restrict extraction to the expected top-level parent folder and reject path traversal.
    with zipfile.ZipFile(ZIP_PATH) as archive:
        members = archive.infolist()
        bad = [m.filename for m in members if not (m.filename == 'NCM Research/' or m.filename.startswith('NCM Research/'))]
        if bad:
            raise RuntimeError(f"Archive contains unexpected root paths: {bad[:5]}")
        for m in members:
            target = (DRIVE_ROOT / m.filename).resolve()
            if DRIVE_ROOT.resolve() not in target.parents and target != DRIVE_ROOT.resolve():
                raise RuntimeError(f"Unsafe archive member: {m.filename}")
        archive.extractall(DRIVE_ROOT)

PROJECT_ROOT = PROJECT_PARENT / 'github'
HF_ROOT = PROJECT_PARENT / 'huggingface'
assert (PROJECT_ROOT / 'src' / 'ncm_factuality').is_dir()
assert (HF_ROOT / 'README.md').is_file()
print('Project parent:', PROJECT_PARENT)
print('GitHub repository root:', PROJECT_ROOT)
print('Local Hugging Face dataset root:', HF_ROOT)

## Install the project and pin upstream source

The next cell installs from the project's dependency file without replacing Colab's CUDA-enabled PyTorch wheel, then checks out the pinned upstream NCM commit. It still does not fetch model checkpoints.

In [ ]:
UPSTREAM_URL = 'https://github.com/HazyResearch/ncm.git'
UPSTREAM_COMMIT = 'c70274273f489abc73832b2b98584a8483b6b203'
UPSTREAM_ROOT = Path('/content/ncm-upstream')
if not (UPSTREAM_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', UPSTREAM_URL, str(UPSTREAM_ROOT)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM_ROOT), 'fetch', 'origin', UPSTREAM_COMMIT], check=True)
subprocess.run(['git', '-C', str(UPSTREAM_ROOT), 'checkout', '--detach', UPSTREAM_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(UPSTREAM_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if actual != UPSTREAM_COMMIT:
    raise RuntimeError(f'Upstream NCM revision mismatch: {actual} != {UPSTREAM_COMMIT}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT), '--no-deps'], check=True)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if str(UPSTREAM_ROOT) not in sys.path:
    sys.path.insert(0, str(UPSTREAM_ROOT))
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA-enabled Colab GPU runtime is required for notebook 01.')
print(torch.cuda.get_device_name(0))
print('Pinned upstream commit:', actual)
# Syntax/import preflight; no checkpoint weights are loaded here.
from models.next_concept import NextConceptTransformer
from tokenization.vqvae_tokenizer import VQVAETokenizer
from ncm_factuality.config import load_config, resolve_paths
cfg = load_config(PROJECT_ROOT / 'configs' / 'experiment.yaml')
paths = resolve_paths(PROJECT_ROOT, cfg)
paths['run_dir'].mkdir(parents=True, exist_ok=True)
print('Run directory:', paths['run_dir'])
print('Preflight passed. This cell did not download or load NCM weights.')

### Before proceeding

If any import, revision, or GPU check failed, stop here. Do not continue to inference with a partially initialized runtime. A runtime restart may be needed if pip changed libraries; after restarting, rerun this notebook from the top.